In [ ]:
# Setting seeds for repeatable results:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import torch, random, os, cv2
import numpy as np

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import seaborn as sns
sns.set_theme(style="whitegrid")

import matplotlib.pyplot as plt

from headset_localization import *

## Loading fr2/desk

In [ ]:

env_fr2desk, rec_fr2desk = scanned_3d_environment_and_headset_recording_from_tum(
        folder="../tum_datasets/rgbd_dataset_freiburg2_desk",
        rgb_camera_name="freiburg2",
        time_tolerance= 0.03,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(iforest_contamination = 0.5, use_depth_images_if_provided=False),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=True),
        intervall=(0.0, 0.06)
)

In [ ]:
visualize_robot_camera_environment_combo(robot_env=env_fr2desk, headset_data=rec_fr2desk, vis_headset_camera_wireframes = False)

## Loading fr2/dishes

In [ ]:
env_fr2dishes, rec_fr2dishes = scanned_3d_environment_and_headset_recording_from_tum(
        folder="../tum_datasets/rgbd_dataset_freiburg2_dishes",
        rgb_camera_name="freiburg2",
        time_tolerance= 0.03,
        n_robot_images= 10,
        xyz_image_generation_config=XYZImageGenerationConfig(iforest_contamination = 0.5, use_depth_images_if_provided=True),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=False),
        intervall=(0.85, 0.88)
)

In [ ]:
visualize_robot_camera_environment_combo(robot_env=env_fr2dishes, headset_data=rec_fr2dishes, vis_headset_camera_wireframes = False)

## Loading fr2/long

In [ ]:
env_fr3long, rec_fr3long = scanned_3d_environment_and_headset_recording_from_tum(
        folder="../tum_datasets/rgbd_dataset_freiburg3_long_office_household",
        rgb_camera_name="freiburg3",
        time_tolerance= 0.03,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(iforest_contamination = 0.5, use_depth_images_if_provided=True),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=False),
        intervall=(0.73,0.78)
)

In [ ]:
visualize_robot_camera_environment_combo(robot_env=env_fr3long, headset_data=rec_fr3long, vis_headset_camera_wireframes = False)

In [ ]:
def predictors_for_dataset(intrinsic_mtx:np.ndarray)->list[GradableLocalizer]:
    points_light_glue = GradableLocalizer(
        creator= PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(),
                rotation_augmentations=[Augmentation]
            )
        ),
        name="PnP-LG"
    )

    points_loma = GradableLocalizer(
        creator= PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
                rotation_augmentations=[Augmentation]
            )
        ),
        name="PnP-LoMa"
    )

    points_lines_light_glue = GradableLocalizer(
        creator=PnPLLocalizer.get_creation_function(
            cam2_intrinsic_mtx = intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(),
                rotation_augmentations=[Augmentation]
            ),
            line_matching_config = LineMatchingConfig(max_point_line_dist_px=20, better_factor=1.2),
            debug_visualize_matching = False,
        ),
        name="PnP+L-LG"
    )

    yolo = YOLOv26Segmenter("yoloe-26l-seg.pt", 
                        prompts=[
                            "monitor", "keyboard", "mouse", "teddy", "tape", "book", "cup", "telephone", "can", "office appliance",
                            "dish", "plate", "scissors", "cup", "bottle", "bowl"
                        ]
                    )

    ellipsoids_light_glue = GradableLocalizer(
        creator=EllipsoidLocalizer.get_creation_function(
            cam2_intrinsic_mtx = intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(),
                rotation_augmentations=[Augmentation]
            ),
            pne_optimizer=PyposePNEOptimizer(),
            cam1_segmenter=yolo,
            cam2_segmenter=yolo,
            matching_config=GaussianMatchingConfig(dummy_value=0.001),
            ellipsoid_matching_config = PointCloudMatchingConfig(min_cluster_size=1, max_color_dist=20),
            ellipsoid_fitter=MVEEEllipsoidFitter(contamination=0.3),
            visualize_environment_generation = True,
            visualize_segmentation_masks = False,
            visualize_matching=False,
            visualize_pne_optimisation=False
        ),
        name="Ellipsoids"
    )

    return [points_loma, points_light_glue, points_lines_light_glue, ellipsoids_light_glue]

def grader_for_dataset(env:Scanned3dEnvironment,headset_recording:HeadsetRecording)->NPredictors1DatasetGrader:
    return NPredictors1DatasetGrader(
        gradable_pose_predictors=predictors_for_dataset(headset_recording.intrinsic_cam_mtx),
        headset_data = headset_recording,
        robot_env = env,
        compute_ray_intersection_error=True, use_tqdm_for_frames=True, use_tqdm_for_predictors=False
    )

def plot_from_grader(grader:NPredictors1DatasetGrader, vis:bool = False):
    if vis:
        grader.visualize_predictions_3d()
    grader.print_summary()
    _, ax = plt.subplots(1, 1, figsize = (12, 3))
    grader.plot_time_series_error(ax, TimeSeriesErrorType.ABS_TRANSLATIONAL)
    _, axes = plt.subplots(2, 3, figsize = (15, 8))
    grader.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]], explain = True)

In [ ]:
grader_fr2desk = grader_for_dataset(env=env_fr2desk, headset_recording=rec_fr2desk)
plot_from_grader(grader_fr2desk)
grader_fr2desk.save_results(name="more_tum_fr2desk")

In [ ]:
grader_fr2dishes = grader_for_dataset(env=env_fr2dishes, headset_recording=rec_fr2dishes)
plot_from_grader(grader_fr2dishes)
grader_fr2dishes.save_results(name="more_tum_fr2dishes")

In [ ]:
grader_fr3long = grader_for_dataset(env=env_fr3long, headset_recording=rec_fr3long)
plot_from_grader(grader_fr3long)
grader_fr3long.save_results(name="more_tum_fr3long")